# 07 — 2026-10-19: coding the explanations with two LLM judges

The explanations of experiment 06 are coded by Claude Haiku 5.5 and GPT-6 Luna (OpenRouter) with the five codes in
`configs/2026-10-19_code_explanations.toml`. Results: `outputs/2026-10-19_code_explanations/judge_labels.jsonl`, one
row per (explanation, judge); finished rows are skipped.

In [1]:
N_QUESTIONS = None  # None = all 598

In [2]:
import hashlib, json, os, threading, tomllib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().parent
CFG = tomllib.loads((ROOT / "configs/2026-10-19_code_explanations.toml").read_text())
SOURCE = tomllib.loads((ROOT / f"configs/{CFG['source_experiment']}.toml").read_text())
CODES = list(CFG["codes"])
JUDGES = CFG["judge_models"]
EXP_DIR = ROOT / "outputs" / CFG["experiment"]
EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT = EXP_DIR / "judge_labels.jsonl"

expl = pd.read_csv(ROOT / "outputs" / CFG["source_experiment"] / "explanations.csv")
options = pd.read_parquet(ROOT / "data-private/processed/emotion_questions.parquet").set_index("question_id")
expl["key"] = expl.question_id + "|" + expl.condition
expl = expl[expl.order.isin(sorted(expl.order.unique())[:N_QUESTIONS])]


def options_text(row):
    texts = {L: options.loc[row.question_id, f"option_{L}"].strip() for L in "ABCD"} | {"E": SOURCE["option_e"]}
    return "\n".join(f"{shown}. {texts[original]}" for shown, original in zip("ABCDE", row.option_order))


expl["options_shown"] = [options_text(r) for r in expl.itertuples()]
expl["answer_text"] = [r.options_shown.split("\n")["ABCDE".index(r.answer_shown)] for r in expl.itertuples()]
print(f"{expl.question_id.nunique()} questions, {len(expl)} explanations; judges: {JUDGES} -> {OUT.relative_to(ROOT)}")

598 questions, 3588 explanations; judges: ['anthropic/claude-haiku-5.5', 'openai/gpt-6-luna'] -> outputs/2026-10-19_code_explanations/judge_labels.jsonl


## Judges

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(ROOT / ".env")
client = OpenAI(base_url=CFG["api_base_url"], api_key=os.environ["OPENROUTER_API_KEY"])
SYSTEM = CFG["prompts"]["system"].format(codes="\n".join(f"- {c}: {d}" for c, d in CFG["codes"].items()))
PROMPT_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()[:12]
SCHEMA = {"type": "object",
          "properties": {"rationale": {"type": "string"}, **{c: {"type": "boolean"} for c in CODES}},
          "required": ["rationale"] + CODES, "additionalProperties": False}


def item_text(row):
    return CFG["prompts"]["item"].format(question=row.question.strip(), options=row.options_shown,
                                         answer=row.answer_text, explanation=row.explanation)


def judge(row, model):
    extra = {"seed": CFG["seed"]} if model in CFG["seed_models"] else {}
    response = client.chat.completions.create(
        model=model, max_tokens=CFG["max_tokens"],
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": item_text(row)}],
        response_format={"type": "json_schema", "json_schema": {"name": "codes", "strict": True, "schema": SCHEMA}},
        extra_body={"reasoning": {"effort": CFG["reasoning_effort"]}}, **extra,
    )
    choice = response.choices[0]
    labels = json.loads(choice.message.content)
    assert set(labels) == {"rationale"} | set(CODES), labels
    details = getattr(response.usage, "completion_tokens_details", None)
    return {"key": row.key, "model": model, **labels, "finish_reason": choice.finish_reason,
            "response_model": response.model, "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "reasoning_tokens": getattr(details, "reasoning_tokens", None), "prompt_sha": PROMPT_SHA}


def done():
    return {(r["key"], r["model"]) for r in map(json.loads, OUT.open())} if OUT.exists() else set()


print(SYSTEM)
print("\n===== example item")
print(item_text(expl.iloc[0]))

You code short explanations written by an AI model. The model answered a multiple-choice question about a person's emotions in a scene from a short film, and was then asked: "Explain in one sentence which information you based your answer on." You are not told what input the model had (it may or may not have had a transcript or a video).

Code only what the explanation itself claims, not whether the claim is true or the answer is right. Apply each code independently; several or none can apply.

Codes:
- cites_dialogue: The explanation quotes or paraphrases what a character says, or refers to a transcript as something the model had (for example "the transcript says ..." or "the transcript does not mention ..."). Subtitles and captions count as a transcript. Saying only that there is no dialogue, or that no transcript was provided, without referring to a transcript the model had, is NOT this code. Do not count speech that only repeats the wording of the question or of any answer option.


## Run

In [4]:
lock, errors = threading.Lock(), []

def work(task):
    row, model = task
    try:
        result = judge(row, model)
    except Exception as e:
        errors.append((row.key, model, repr(e)[:200]))
        return
    with lock, OUT.open("a") as f:
        f.write(json.dumps(result) + "\n")

finished = done()
tasks = [(row, model) for model in JUDGES for row in expl.itertuples() if (row.key, model) not in finished]
with ThreadPoolExecutor(CFG["n_workers"]) as pool:
    list(tqdm(pool.map(work, tasks), total=len(tasks), unit="label"))
print(f"done {len(tasks) - len(errors)}, failed {len(errors)}")
for e in errors[:10]:
    print(" ", *e)

  0%|          | 0/7176 [00:00<?, ?label/s]

done 7176, failed 0


## Check

In [5]:
labels = pd.read_json(OUT, lines=True)
labels = labels[labels.key.isin(expl.key)]
display(labels.groupby("model")[["finish_reason", "reasoning_tokens"]].agg(lambda s: s.value_counts(dropna=False).to_dict()))
labels["codes"] = labels[CODES].apply(lambda r: ", ".join(c for c in CODES if r[c]) or "-", axis=1)
side = labels.pivot(index="key", columns="model", values="codes")
side.columns = [m.split("/")[1] for m in side.columns]
pd.set_option("display.max_colwidth", None)
display(expl.head(30).set_index("key")[["explanation"]].join(side))

,finish_reason,reasoning_tokens
model,,
anthropic/claude-haiku-5.5,{'stop': 3588},"{0: 1879, 204: 14, 173: 14, 201: 14, 152: 14, ..."
openai/gpt-6-luna,{'stop': 3588},"{72: 39, 104: 37, 92: 36, 76: 34, 66: 34, 78: ..."


,explanation,claude-haiku-5.5,gpt-6-luna
key,,,
FH-76|Q,The answer is based on the information that there is no specific emotion or sentiment expressed by the man regarding knocking on the door in the given context.,says_insufficient,says_insufficient
FH-76|Q_noE,"The answer is based on the man expressing fear that there might not be anyone inside, which indicates uncertainty and potential disappointment.",-,-
FH-76|T,"The information is based on the fact that the man is knocking on the door, which implies that he is trying to get someone's attention, but there is no dialogue to indicate his feelings about it.",says_insufficient,says_insufficient
FH-76|T_told,"The answer is based on the fact that there is no dialogue provided in the transcript, making it impossible to determine the man's feelings about knocking on the door.","cites_dialogue, says_insufficient","cites_dialogue, says_insufficient"
FH-76|VT,The man expresses fear that there might not be anyone inside.,-,-
FH-76|VTmis,The man expresses fear that there might not be anyone inside.,-,-
97rJU|Q,"The answer is based on the information that the young woman steps back, indicating she feels scared and wants to avoid further violence.",-,-
97rJU|Q_noE,"The answer is based on the information that the young woman steps back, indicating her fear of being hurt again.",-,-
97rJU|T,The information is based on the young woman's action of stepping back after the older woman's violent reaction.,-,-


## Status

In [6]:
labels = pd.read_json(OUT, lines=True)
status = labels.groupby("model").agg(rows=("key", "nunique"), prompt_tokens=("prompt_tokens", "sum"),
                                      completion_tokens=("completion_tokens", "sum"))
status["cost_usd"] = (status.prompt_tokens * 0.10 + status.completion_tokens * 0.50) / 1e6
print(f"{len(expl)} explanations; prompt versions: {labels.prompt_sha.value_counts().to_dict()}")
display(status)

3588 explanations; prompt versions: {77044885371: 7176}


,rows,prompt_tokens,completion_tokens,cost_usd
model,,,,
anthropic/claude-haiku-5.5,3588,6232362,946253,1.096363
openai/gpt-6-luna,3588,3613147,816275,0.769452


## Agreement between the judges

In [ ]:
from sklearn.metrics import cohen_kappa_score

labels = pd.read_json(OUT, lines=True, dtype={"prompt_sha": str})
labels = labels[labels.key.isin(expl.key)]
wide = {m: labels[labels.model == m].set_index("key")[CODES].astype(bool).reindex(expl.key) for m in JUDGES}
a, b = wide[JUDGES[0]], wide[JUDGES[1]]
agreement = pd.DataFrame({"agree": (a == b).mean(), "kappa": [cohen_kappa_score(a[c], b[c]) for c in CODES],
                          **{f"yes_{m.split('/')[1]}": wide[m].sum() for m in JUDGES}})
agreement.to_csv(EXP_DIR / "agreement.csv")
print("explanations with any disagreement:", (a != b).any(axis=1).sum(), "of", len(expl))
agreement.round(3)

## Codes by condition

% of explanations where both judges give the code.

In [ ]:
conditions = [c for c in SOURCE["conditions"] if c in set(expl.condition)]
condition = expl.condition.values
by_condition = (a & b).groupby(condition).mean().reindex(conditions) * 100
by_condition["judges_differ"] = (a != b).any(axis=1).groupby(condition).mean().reindex(conditions) * 100
by_condition.to_csv(EXP_DIR / "codes_by_condition.csv")
audio = expl.explanation.str.contains(r"\baudio", case=False)
print('explanations mentioning "audio":', audio.groupby(condition).sum().reindex(conditions).to_dict())
by_condition.round(1)